# Train segmentation vùng giả mạo trên Kaggle

Trước khi chạy (panel bên phải, mục **Settings**):
- **Accelerator**: GPU P100 (hoặc GPU T4 x2, code chỉ dùng 1 GPU)
- **Internet**: On (để clone code từ GitHub và tải checkpoint `openmmlab/upernet-swin-tiny`)
- **Input**: Add Input → Your Datasets → `chituok/ffpp-faces-c23` (private)

Bấm **Save Version → Save & Run All (Commit)** để chạy nền (tắt trình duyệt vẫn chạy).
Mỗi phiên tối đa 12 giờ, nên train tự dừng sau khoảng 11 giờ, ở cuối một epoch.
Kết quả nằm ở tab **Output**: `runs/<tên run>/`.

**Train tiếp ở phiên sau:** tải `last.pt`, `best.pt`, `history.jsonl` của phiên trước lên
một dataset riêng tư (ví dụ `chituok/swin-upernet-ckpt`), Add Input dataset đó, rồi đặt
`RESUME` là tên dataset. Chạy lại cho tới khi đủ số epoch trong config.


In [ ]:
CONFIG = 'configs/swin_upernet.yaml'
EXTRA = ''                     # ví dụ: '--epochs 1 --fraction 0.05' để chạy thử
RESUME = ''                    # tên dataset chứa last.pt của phiên trước, '' = train mới


In [ ]:
import glob, os, zipfile

# Kaggle thường tự giải nén file zip khi tạo dataset; nếu không thì giải nén vào /tmp
# chỉ dò 3 tầng đầu: quét đệ quy cả 287k ảnh trên ổ mạng của Kaggle mất nhiều phút
found = [f for d in ('*', '*/*', '*/*/*') for f in glob.glob('/kaggle/input/' + d + '/index/train.csv')]
if not found:
    z = glob.glob('/kaggle/input/**/*.zip', recursive=True)[0]
    zipfile.ZipFile(z).extractall('/tmp/data')
    found = ['/tmp/data/index/train.csv']
INDEX = os.path.dirname(found[0])
FACES = os.path.join(os.path.dirname(INDEX), 'faces')
print(INDEX, FACES, os.path.isdir(FACES))

# /kaggle/input là ổ mạng, mỗi ảnh chờ ~0,25 s nên GPU đứng chờ dữ liệu (~1 s/bước):
# chép một lần sang đĩa máy bằng nhiều luồng song song
import csv, shutil, time
from concurrent.futures import ThreadPoolExecutor
files = sorted({r[k] for s in ('train', 'val', 'test')
                for r in csv.DictReader(open(f'{INDEX}/{s}.csv', encoding='utf-8'))
                for k in ('frame', 'mask') if r[k]})
def copy(rel):
    dst = '/tmp/faces/' + rel
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copyfile(os.path.join(FACES, rel), dst)
t = time.time()
with ThreadPoolExecutor(128) as ex:
    for i, _ in enumerate(ex.map(copy, files), 1):
        if i % 20000 == 0 or i == len(files):
            print(f'chép {i}/{len(files)} ảnh, {time.time() - t:.0f} s', flush=True)
FACES = '/tmp/faces'
os.environ['TQDM_MININTERVAL'] = '60'   # log thanh tiến trình mỗi phút, tránh log quá dài

In [ ]:
!nvidia-smi
!rm -rf /tmp/code && git clone -q https://github.com/nguyenchitu2005-jpg/DoAnChuyenNganh.git /tmp/code
# cùng phiên bản với máy nhà để checkpoint nạp lại được khi đánh giá
!pip install -q segmentation-models-pytorch==0.5.0 albumentations==2.0.8 transformers==5.19.0


In [ ]:
%cd /tmp/code
if RESUME:   # input chỉ đọc: chép checkpoint sang working/runs/<tên run> rồi train tiếp
    import yaml
    SRC = [p for d in ('.', '*', '*/*') for p in glob.glob(f'/kaggle/input/{d}/{RESUME}')][0]
    RUN_DIR = '/kaggle/working/runs/' + yaml.safe_load(open(CONFIG))['name']
    !mkdir -p {RUN_DIR} && cp {SRC}/* {RUN_DIR}/ && ls -la {RUN_DIR}
    EXTRA += ' --resume'
!python src/train.py --config {CONFIG} --data-root {FACES} --index-dir {INDEX} --num-workers 4 --runs-dir /kaggle/working/runs --max-hours 11 {EXTRA}


In [ ]:
# Đánh giá luôn trên FF++ test (DFD và c40 đánh giá ở máy nhà)
for d in glob.glob('/kaggle/working/runs/*'):
    if os.path.exists(d + '/best.pt'):
        !python src/evaluate.py {d}/best.pt --split test